<a href="https://colab.research.google.com/github/ekosup/pkn-stan-padaa/blob/main/basic/sql_basic_refresh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Basic — Refreshment Sebelum Audit Data - Cek GH

Notebook ini me-refresh konsep SQL dasar yang akan dipakai di modul audit.
Fokus: **kamu menulis SQL, Python cuma alat bantu eksekusi.**

> Estimasi: 45 menit (8 topik)
> Koneksi: NeonDB (PostgreSQL)

### Cara pakai
1. Isi `NEON_CONN` di sel SETUP dengan connection string
2. Jalankan sel SETUP
3. Pelajari tiap topik.
4. Di sel **Eksplorasi Bebas** kamu bisa coba query sendiri
5. Tambahkan sel baru jika diperlukan

---
## SETUP — Jalankan sekali

In [ ]:
# !pip install psycopg2-binary -q

In [ ]:
import time
import pandas as pd
import psycopg2
from google.colab import userdata
from contextlib import contextmanager

NEON_CONN = userdata.get("NEON_CONN")


class Database:
    def __init__(self, conn_string):
        self.conn = psycopg2.connect(conn_string)

    def close(self):
        self.conn.close()

    def run(self, sql, params=None):
        start = time.perf_counter()

        with self.conn.cursor() as cur:
            try:
                cur.execute(sql, params)

                elapsed = time.perf_counter() - start

                if cur.description:
                    cols = [c.name for c in cur.description]
                    rows = cur.fetchall()

                    print(f"✅ {len(rows)} row(s) ({elapsed:.3f}s)")
                    return pd.DataFrame(rows, columns=cols)

                self.conn.commit()
                print(f"✅ {cur.rowcount} row(s) affected ({elapsed:.3f}s)")

            except Exception as e:
                self.conn.rollback()
                print(f"❌ SQL Error:\n{e}")
                raise

    def tables(self):
        return self.run("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema='public'
            ORDER BY table_name
        """)

    def describe(self, table):
        return self.run("""
            SELECT
                column_name,
                data_type,
                is_nullable
            FROM information_schema.columns
            WHERE table_name=%s
            ORDER BY ordinal_position
        """, (table,))

    def schema(self):
        return self.run("""
            SELECT
                table_name,
                column_name,
                data_type
            FROM information_schema.columns
            WHERE table_schema='public'
            ORDER BY table_name, ordinal_position
        """)

    def explain(self, sql):
        return self.run(f"EXPLAIN ANALYZE {sql}")


db = Database(NEON_CONN)

def q(sql):
    return db.run(sql)

def schema():
    return db.schema()

def tables():
    return db.tables()

def desc(table):
    return db.describe(table)

print("✅ Database siap!")

### 🔍 Cek tabel yang tersedia

In [ ]:
tables()

---
## 1️⃣ SELECT & FROM — Lihat isi tabel

`SELECT` = kolom apa yang ingin dilihat. `FROM` = dari tabel mana.
`*` artinya "semua kolom". `LIMIT` membatasi jumlah baris.

In [ ]:
# Semua kolom, 5 baris pertama
q("SELECT * FROM employees LIMIT 5")

In [ ]:
# Hanya nama & gaji
q("SELECT emp_name, salary FROM employees")

---
## 2️⃣ WHERE — Menyaring baris

`WHERE` = saringan kopi. Hanya baris yang lolos kriteria yang diteruskan.

Operator: `=`, `<>` (tidak sama), `>`, `<`, `>=`, `<=`

In [ ]:
# Karyawan dengan gaji di atas 20 juta
q("SELECT emp_name, salary FROM employees WHERE salary > 20000000")

In [ ]:
# Karyawan yang SUDAH KELUAR (is_active = false)
q("SELECT emp_name, hire_date, is_active FROM employees WHERE is_active = false")

In [ ]:
# Kombinasi: gaji >= 20jt DAN di departemen 101
q("SELECT emp_name, salary, dept_id FROM employees WHERE salary >= 20000000 AND dept_id = 101")

### 🧪 Eksplorasi Bebas — WHERE

In [ ]:
# Tulis query WHERE-mu sendiri di sini!
# Misal: cari karyawan yang gajinya di antara 15jt dan 25jt

q("""
    -- ✏️ TULIS QUERY DI SINI
    SELECT emp_name, salary FROM employees WHERE salary BETWEEN 15000000 AND 25000000;
""")

---
## 3️⃣ ORDER BY & LIMIT — Urutkan & batasi

`ORDER BY kolom ASC/DESC` — urutkan naik/turun.
`LIMIT n` — ambil n baris pertama saja.

In [ ]:
# 3 karyawan dengan gaji tertinggi
q("SELECT emp_name, salary FROM employees ORDER BY salary DESC LIMIT 3")

In [ ]:
# Proyek dengan budget terbesar (aktif saja)
q("SELECT proj_name, budget FROM projects WHERE status='aktif' ORDER BY budget DESC")

---
## 4️⃣ GROUP BY & Agregasi — Ringkas data

Agregat: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`.
`GROUP BY` = kelompokkan dulu baru dihitung.

> Analogi: mengelompokkan uang receh per nominal sebelum menghitung total.

In [ ]:
# Jumlah karyawan per departemen
q("SELECT dept_id, COUNT(*) AS jumlah_karyawan FROM employees GROUP BY dept_id ORDER BY jumlah_karyawan DESC")

In [ ]:
# Rata-rata gaji per departemen (hanya departemen dengan rata-rata > 15jt)
q('''SELECT dept_id, ROUND(AVG(salary), 0) AS rata_gaji, COUNT(*) AS jumlah
       FROM employees
       GROUP BY dept_id
       HAVING AVG(salary) > 15000000
       ORDER BY rata_gaji DESC''')

**Penting:** `WHERE` menyaring SEBELUM agregasi. `HAVING` menyaring SESUDAH agregasi.

In [ ]:
# Bandingkan: WHERE dulu baru GROUP BY
# Total gaji per dept, hanya karyawan AKTIF
q('''SELECT dept_id, SUM(salary) AS total_gaji
       FROM employees
       WHERE is_active = true
       GROUP BY dept_id
       ORDER BY total_gaji DESC''')

### 🧪 Eksplorasi Bebas — Agregasi

In [ ]:
# Coba hitung total budget proyek per status!
q("""SELECT e.dept_id, d.dept_name, SUM(e.salary) AS total_gaji
  FROM employees e
  JOIN departments d
    ON e.dept_id = d.dept_id
  WHERE e.is_active = true
  GROUP BY e.dept_id, d.dept_name
  ORDER BY total_gaji DESC;""")

---
## 5️⃣ BETWEEN, IN, LIKE — Operator lanjutan

- `BETWEEN x AND y` — rentang (inklusif)
- `IN (a, b, c)` — salah satu dari daftar
- `LIKE 'pola%'` — pencocokan teks (`%` = wildcard)

In [ ]:
# Gaji antara 15jt dan 25jt
q("SELECT emp_name, salary FROM employees WHERE salary BETWEEN 15000000 AND 25000000 ORDER BY salary")

In [ ]:
# Departemen 101, 103, atau 105
q("SELECT emp_name, dept_id FROM employees WHERE dept_id IN (101, 103, 105) ORDER BY dept_id")

In [ ]:
# Nama depan berawalan huruf 'A'
q("SELECT emp_name FROM employees WHERE emp_name LIKE 'A%'")

---
## 6️⃣ JOIN — Gabungkan dua tabel

`INNER JOIN` = hanya yang cocok di kedua sisi.
`LEFT JOIN`  = semua dari kiri + yang cocok dari kanan (NULL kalau tidak ada).

In [ ]:
# INNER JOIN: karyawan + nama departemennya
q('''SELECT e.emp_name, e.salary, d.dept_name
       FROM employees e
       JOIN departments d ON e.dept_id = d.dept_id
       ORDER BY e.salary DESC''')

In [ ]:
# LEFT JOIN: SEMUA karyawan, termasuk yang TIDAK PUNYA departemen
# (Lina, dept_id=NULL — lihat bedanya dengan INNER JOIN!)
q('''SELECT e.emp_name, e.dept_id, d.dept_name
       FROM employees e
       LEFT JOIN departments d ON e.dept_id = d.dept_id
       ORDER BY e.emp_id''')

In [ ]:
# LEFT JOIN dimanfaatkan: cari yang TIDAK PUNYA pasangan
q('''SELECT e.emp_name, e.dept_id
       FROM employees e
       LEFT JOIN departments d ON e.dept_id = d.dept_id
       WHERE d.dept_id IS NULL''')

### 🧪 Eksplorasi Bebas — JOIN

In [ ]:
# Gabungkan projects + departments: tampilkan nama proyek + nama departemen
q("""
    -- ✏️ TULIS QUERY DI SINI
    SELECT p.proj_name, d.dept_name
    FROM projects p
    JOIN departments d ON p.dept_id = d.dept_id;

""")

---
## 7️⃣ NULL & COALESCE — Tangani data kosong

`NULL` ≠ 0 dan ≠ `''` (string kosong). NULL artinya "tidak diketahui".
`COALESCE(a, b, c)` = ambil nilai pertama yang bukan NULL.

> Analogi: nomor telepon utama vs nomor darurat.

In [ ]:
# Siapa yang tidak punya departemen?
q("SELECT emp_name, dept_id FROM employees WHERE dept_id IS NULL")

In [ ]:
# COALESCE: ganti NULL dengan label bermakna
q('''SELECT emp_name,
              COALESCE(dept_id::TEXT, 'TANPA DEPARTEMEN') AS departemen
       FROM employees ORDER BY emp_id''')

---
## 8️⃣ Subquery — Query di dalam query

Subquery = kumpulkan daftar dulu (dalam), baru pakai daftar itu (luar).

> Analogi: detektif kumpulkan daftar tersangka, baru menyaring TKP dengan daftar itu.

In [ ]:
# Karyawan yang gajinya DI ATAS rata-rata semua karyawan
q('''SELECT emp_name, salary
       FROM employees
       WHERE salary > (SELECT AVG(salary) FROM employees)
       ORDER BY salary DESC''')

In [ ]:
# Proyek dari departemen yang punya karyawan bergaji > 25jt
q('''SELECT proj_name, dept_id, budget
       FROM projects
       WHERE dept_id IN (
           SELECT DISTINCT dept_id FROM employees WHERE salary > 25000000
       )
       ORDER BY budget DESC''')

---
## 🧪 Latihan Gabungan — Uji semua konsep

Kerjakan 3 soal pendek. Tulis query di sel masing-masing.

### Soal A
Tampilkan **nama karyawan, nama departemen, gaji** untuk semua karyawan AKTIF.
Urutkan berdasarkan gaji tertinggi.

In [ ]:
q("""
    -- ✏️ TULIS QUERY DI SINI

""")

### Soal B
Hitung **total gaji per departemen**, tapi hanya untuk departemen dengan **total > 40jt**.
Tampilkan nama departemen dan total gaji.

In [ ]:
q("""
    -- ✏️ TULIS QUERY DI SINI

""")

### Soal C
Tampilkan proyek-proyek yang budgetnya **di atas rata-rata budget semua proyek**.
Tampilkan nama proyek, budget, dan status.

In [ ]:
q("""
    -- ✏️ TULIS QUERY DI SINI

""")

---
## ✅ Selesai!

Konsep yang sudah di-refresh:

| Topik | Keyword SQL | Akan dipakai di Modul Audit |
|-------|-----------|---------------------------|
| SELECT, FROM, LIMIT | `SELECT`, `FROM`, `LIMIT` | 2.1 — Profiling |
| WHERE, AND, OR | `WHERE`, `AND`, `OR` | 2.3 — Filter anomali |
| ORDER BY | `ORDER BY`, `ASC`, `DESC` | Semua modul |
| GROUP BY, agregat | `GROUP BY`, `SUM`, `AVG`, `COUNT`, `HAVING` | 3.1 — Materialitas |
| BETWEEN, IN, LIKE | `BETWEEN`, `IN`, `LIKE` | 2.3 — Filter |
| JOIN (INNER, LEFT) | `JOIN`, `LEFT JOIN` | 2.1 — Akun hantu, 3.2 — Siphoning |
| NULL, COALESCE | `IS NULL`, `COALESCE` | 2.2 — Cleansing |
| Subquery | `IN (SELECT ...)`, subquery skalar | 3.2 — Investigasi |

Lanjut ke **Notebook Audit Data** untuk menerapkan semua ini ke data General Ledger.